In [ ]:
import pandas as pd
import altair as alt
from vega_datasets import data


### Visualization 1: Quarterly Prescription Volume: Alprazolam vs. Lorazepam

This visualization compares the total number of Medicaid prescriptions for Alprazolam and Lorazepam across the four quarters of 2024. It helps identify which medication has higher utilization and whether prescription patterns change throughout the year.

In [21]:
# load cleaned dataset
df = pd.read_csv("official.csv")

# keep only 2024 data
df = df[df["Year"] == 2024].copy()

# standardize medication and state names
df["Product Name"] = df["Product Name"].str.strip().str.title()
df["State"] = df["State"].str.strip().str.upper()

# calculate total prescriptions by quarter and medication
quarterly = (
    df.groupby(["Quarter", "Product Name"], as_index=False)
    ["Number of Prescriptions"].sum()
)

# create grouped bar chart
chart = alt.Chart(quarterly).mark_bar(
    cornerRadiusTopLeft=4,
    cornerRadiusTopRight=4,
    size=32
).encode(
    # horizontal quarter labels
    x=alt.X(
        "Quarter:O",
        title="Quarter (2024)",
        axis=alt.Axis(labelAngle=0, labelPadding=12, tickSize=0)
    ),

    # place bars side by side
    xOffset="Product Name:N",

    # format prescription counts
    y=alt.Y(
        "Number of Prescriptions:Q",
        title="Total Prescriptions",
        axis=alt.Axis(format="~s", grid=True, tickCount=5)
    ),

    # set colors
    color=alt.Color(
        "Product Name:N",
        title="Medication",
        scale=alt.Scale(
            domain=["Alprazolam", "Lorazepam"],
            range=["#376A80", "#A15D83"]
        ),
        legend=alt.Legend(
            orient="top",
            direction="horizontal",
            title=None
        )
    )
).properties(
    width=650,
    height=370,
    title=alt.TitleParams(
        text="Quarterly Medicaid Prescription Volume",
        subtitle="Alprazolam vs. Lorazepam | 2024",
        anchor="middle",
        fontSize=18,
        subtitleFontSize=12,
        offset=18
    )
).configure_axis(
    labelFontSize=12,
    titleFontSize=13,
    titlePadding=15,
    domain=False,
    gridColor="#EAEAEA"
).configure_view(
    stroke=None
)

chart

alt.Chart(...)

### Visualization 3: Medicaid Prescription Volume by State: Alprazolam vs. Lorazepam

This interactive choropleth map compares the total number of Medicaid prescriptions for Alprazolam and Lorazepam across U.S. states in 2024. Users can switch between medications using a dropdown menu and hover over states to view their prescription totals. The visualization helps identify geographic differences in prescription volume and highlights which states have higher utilization for each medication. White states indicate a lack of reported data.

In [20]:
# map state abbreviations to geographic ids
states = {
    "AL": 1, "AK": 2, "AZ": 4, "AR": 5, "CA": 6,
    "CO": 8, "CT": 9, "DE": 10, "DC": 11, "FL": 12,
    "GA": 13, "HI": 15, "ID": 16, "IL": 17, "IN": 18,
    "IA": 19, "KS": 20, "KY": 21, "LA": 22, "ME": 23,
    "MD": 24, "MA": 25, "MI": 26, "MN": 27, "MS": 28,
    "MO": 29, "MT": 30, "NE": 31, "NV": 32, "NH": 33,
    "NJ": 34, "NM": 35, "NY": 36, "NC": 37, "ND": 38,
    "OH": 39, "OK": 40, "OR": 41, "PA": 42, "RI": 44,
    "SC": 45, "SD": 46, "TN": 47, "TX": 48, "UT": 49,
    "VT": 50, "VA": 51, "WA": 53, "WV": 54, "WI": 55,
    "WY": 56
}

df["id"] = df["State"].map(states)

# remove unmappable locations
df = df.dropna(subset=["id"])
df["id"] = df["id"].astype(int)

# calculate prescriptions by state and medication
state_data = df.pivot_table(
    index=["id", "State"],
    columns="Product Name",
    values="Number of Prescriptions",
    aggfunc="sum"
).reset_index()

state_data.columns.name = None

# create medication dropdown
medication = alt.param(
    name="Medication",
    value="Alprazolam",
    bind=alt.binding_select(
        options=["Alprazolam", "Lorazepam"],
        name="Medication: "
    )
)

# load us state boundaries
us_states = alt.topo_feature(data.us_10m.url, "states")

# create interactive map
chart = (
    alt.Chart(us_states)
    .transform_lookup(
        lookup="id",
        from_=alt.LookupData(
            state_data,
            "id",
            ["State", "Alprazolam", "Lorazepam"]
        )
    )
    .transform_calculate(
        Prescriptions=(
            "Medication === 'Alprazolam' "
            "? datum.Alprazolam : datum.Lorazepam"
        )
    )
    .transform_filter("isValid(datum.Prescriptions)")
    .mark_geoshape(
        stroke="white",
        strokeWidth=0.8
    )
    .encode(
        color=alt.Color(
            "Prescriptions:Q",
            title="Total Prescriptions",
            scale=alt.Scale(
                scheme="tealblues",
                domain=[0, 200000]
            ),
            legend=alt.Legend(
                format="~s",
                titleFontSize=12,
                labelFontSize=11
            )
        ),
        tooltip=[
            alt.Tooltip("State:N", title="State"),
            alt.Tooltip(
                "Prescriptions:Q",
                title="Total Prescriptions",
                format=","
            )
        ]
    )
    .add_params(medication)
    .project(type="albersUsa")
    .properties(
        width=650,
        height=400
    )
)

# add bottom-right note
note = (
    alt.Chart(
        pd.DataFrame({
            "text": ["White states indicate no reported data"]
        })
    )
    .mark_text(
        align="right",
        baseline="middle",
        fontSize=11,
        fontStyle="italic",
        color="gray"
    )
    .encode(
        x=alt.value(645),
        y=alt.value(12),
        text="text:N"
    )
    .properties(
        width=650,
        height=25
    )
)

# combine map and note
final_chart = (
    alt.vconcat(
        chart,
        note,
        spacing=0
    )
    .properties(
        title=alt.TitleParams(
            text="Medicaid Prescription Volume by State",
            subtitle="Alprazolam vs. Lorazepam | 2024",
            anchor="middle",
            fontSize=18,
            subtitleFontSize=12,
            offset=20
        )
    )
    .configure_view(stroke=None)
)

final_chart

alt.VConcatChart(...)